# Fuzzy Control with `fuzzy_torch.envs`

In notebook 01 we built fuzzy systems and trained them on data. Here we put them to
work as **controllers** for three benchmark plants:

| case study | plant | what the controller does |
|---|---|---|
| **Overhead crane** | `CranePlant` | move a trolley 5 m without letting the payload swing |
| **Fuzzy Grand Prix** | `RacingEnv` (differential-drive kart) | finish a lap as fast as possible without leaving the track |
| **Quadrotor** | `QuadrotorPlant` | fly to and hover at a 3-D target |

All plants are **batched, differentiable torch models** behind the standard
**Gymnasium** API (`reset` / `step`). The same plant therefore serves classical
controllers, fuzzy controllers, RL agents and gradient-based tuning.

> Live 3-D viewers (MuJoCo / pygame) open a window, so they are run as scripts on your
> own machine, e.g. `python examples/racing_demo.py --live`. In this notebook we run
> everything headless and plot with matplotlib.

## 0. Setup

The environments need the `[envs]` extra (gymnasium, numpy, pygame-ce). On Colab the
cell below installs it and fetches the repository for the example files; locally it
uses your checkout.

In [ ]:
import sys, pathlib, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install "torch-fis[envs] @ git+https://github.com/mdrs-thiago/fuzzy_torch.git" matplotlib
    !git clone -q --depth 1 https://github.com/mdrs-thiago/fuzzy_torch.git /content/fuzzy_torch_repo
    REPO = pathlib.Path('/content/fuzzy_torch_repo')
else:
    # This notebook lives in fuzzy_torch/examples/notebooks/.
    REPO = pathlib.Path.cwd().resolve().parents[1]
    try:
        importlib.import_module('fuzzy_torch')
    except ModuleNotFoundError:
        sys.path.insert(0, str(REPO.parent))   # flat package: put its parent on sys.path

import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import fuzzy_torch as ft

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('fuzzy_torch', ft.__version__, '| repo at', REPO)

In [ ]:
from fuzzy_torch import FuzzyVariable, FuzzyRule, FuzzyModule, TriangularMF, TrapezoidalMF, build_system
from fuzzy_torch.envs import (
    CranePlant, QuadrotorPlant, ControlMode, Scenario,
    step_response, disturbance_rejection, step_response_metrics, disturbance_metrics,
    FuzzyControlEnv, ErrorFeatures, RacingEnv,
)
from fuzzy_torch.envs.controllers import crane_baseline, FuzzyRaceDriver, ConfigRaceDriver

## 1. Anatomy of an environment

Every benchmark has the same layers:

```
 Plant                -> physics: state, derivatives(state, u), bounds  (pure torch)
 Scenario             -> episode length, setpoint, scripted disturbances
 FuzzyControlEnv      -> Gymnasium wrapper: reset(), step(action), integrator
 ErrorFeatures        -> optional wrapper: raw state -> [error, error_dot, ...]
 your controller      -> obs -> action
```

Let's look at the crane plant.

In [ ]:
crane = CranePlant()
print('state      :', crane.state_names)
print('tracked    :', crane.tracked_states)
print('actions    :', crane.action_names, '| bounds', [b.tolist() for b in crane.action_bounds()])
print('default dt :', crane.default_dt, 's')

env = FuzzyControlEnv(crane, scenario=step_response(500, setpoint=[5.0, 0.0]))
obs, info = env.reset(seed=0)
print('\nraw observation :', obs)
obs, reward, terminated, truncated, info = env.step([10.0])   # push with 10 N
print('after one step  :', obs)
print('info keys       :', list(info))

The raw state is physical (position, velocity, angle, ...). For control we usually want
**errors**. `ErrorFeatures` turns the state into `[e, ė]` for each tracked state, here
`[x_error, x_error_dot, theta_error, theta_error_dot]`.

In [ ]:
wrapped = ErrorFeatures(FuzzyControlEnv(CranePlant(), scenario=step_response(500, setpoint=[5.0, 0.0])))
obs, _ = wrapped.reset(seed=0)
print(wrapped.feature_names)
print(obs)

## 2. Overhead crane

A trolley carries a payload on a cable. Moving the trolley makes the payload swing;
the controller must reach the 5 m setpoint **and** damp the sway. We first run the
provided **state-feedback baseline**, $F = K \cdot [e_x, \dot e_x, e_\theta, \dot e_\theta]$.

The helper below runs one episode and logs everything we want to plot. It is the same
loop as `examples/crane_demo.py`.

In [ ]:
def crane_rollout(scenario, controller):
    env = ErrorFeatures(FuzzyControlEnv(CranePlant(), scenario=scenario))
    obs, _ = env.reset(seed=0)
    if hasattr(controller, 'reset'):
        controller.reset()
    log = {k: [] for k in ('t', 'x', 'theta', 'force', 'applied', 'target', 'disturbed')}
    for _ in range(scenario.episode_steps):
        obs, _, _, _, info = env.step(controller(obs))
        state = env.unwrapped._state[0]
        log['t'].append(info['time'])
        log['x'].append(float(state[0]))
        log['theta'].append(float(state[2]))
        log['force'].append(float(info['control'][0]))
        log['applied'].append(float(info['applied'][0]))
        log['target'].append(float(info['setpoint'][0]))
        log['disturbed'].append(info['disturbance'] is not None)
    env.close()
    return {k: np.asarray(v) for k, v in log.items()}


def plot_crane(logs, title):
    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    for name, log in logs.items():
        axes[0].plot(log['t'], log['x'], label=name)
        axes[1].plot(log['t'], np.degrees(log['theta']), label=name)
        axes[2].plot(log['t'], log['force'], label=name)
    log = next(iter(logs.values()))
    axes[0].plot(log['t'], log['target'], 'k--', lw=1, label='setpoint')
    if log['disturbed'].any():
        t0, t1 = log['t'][log['disturbed']][[0, -1]]
        for ax in axes:
            ax.axvspan(t0, t1, color='orange', alpha=0.3)
    axes[0].set_ylabel('trolley x [m]'); axes[1].set_ylabel('sway [deg]')
    axes[2].set_ylabel('force [N]'); axes[2].set_xlabel('time [s]')
    axes[0].legend(fontsize=8); axes[0].set_title(title)
    plt.tight_layout()


def crane_report(log, dt=CranePlant.default_dt):
    m = step_response_metrics(torch.tensor(log['x']), target=5.0, dt=dt, initial=0.0,
                              control=torch.tensor(log['force'])).as_dict()
    m['max_sway_deg'] = float(np.degrees(np.abs(log['theta']).max()))
    return m


def show_table(columns):
    """Print {column: {metric: value}} as a small aligned table."""
    names = list(columns)
    rows = list(next(iter(columns.values())))
    print(f"{'':20s}" + ''.join(f'{n:>26s}' for n in names))
    for r in rows:
        print(f'{r:20s}' + ''.join(f'{float(columns[n][r]):26.3f}' for n in names))

STEP = step_response(1500, setpoint=[5.0, 0.0])      # 15 s
baseline = crane_baseline()
base_log = crane_rollout(STEP, baseline)
plot_crane({'state feedback': base_log}, 'Crane step response - baseline')
show_table({'state feedback': crane_report(base_log)})

Controllers are compared with **classical metrics** (rise time, overshoot, settling
time, IAE/ISE/ITAE, control effort), not with the RL reward - that is what makes PID,
fuzzy and RL controllers comparable.

## 3. A hand-written fuzzy crane controller

Now our own controller: two small Mamdani systems added together.

- **Position** FIS: inputs `x_error`, `x_error_dot` → force that drives the trolley.
- **Anti-sway** FIS: inputs `theta_error`, `theta_error_dot` → correction force.
  Note the sign: to damp a pendulum, the trolley must move *towards* the swing.

We use 3 terms per input (`N`egative, `Z`ero, `P`ositive) and the same 3×3 rule table
(a classic "PD-like" fuzzy table) for both. The universe of each input is `[-span, span]`.

In [ ]:
def nzp(name, span):
    v = FuzzyVariable(name, -span, span)
    v.add_term('N', TrapezoidalMF(-span, -span, -span / 2, 0))
    v.add_term('Z', TriangularMF(-span / 2, 0, span / 2))
    v.add_term('P', TrapezoidalMF(0, span / 2, span, span))
    return v

def output_var(name, span):
    v = FuzzyVariable(name, -span, span)
    for term, (a, b, c) in {'NB': (-span, -span, -span / 2), 'NS': (-span, -span / 2, 0),
                            'Z': (-span / 2, 0, span / 2),
                            'PS': (0, span / 2, span), 'PB': (span / 2, span, span)}.items():
        v.add_term(term, TriangularMF(a, b, c))
    return v

#            e_dot:  N     Z     P
PD_TABLE = {'N':   ['NB', 'NS', 'Z'],
            'Z':   ['NS', 'Z',  'PS'],
            'P':   ['Z',  'PS', 'PB']}

def pd_fis(e, de, out):
    rules = [FuzzyRule([(e, te), (de, td)], (out, PD_TABLE[te][j]))
             for te in 'NZP' for j, td in enumerate('NZP')]
    return FuzzyModule([e, de], out, rules, no_rule_value=0.0)

position_fis = pd_fis(nzp('x_error', 3.0), nzp('x_error_dot', 1.5), output_var('F_pos', 30.0))
sway_fis     = pd_fis(nzp('theta_error', 0.2), nzp('theta_error_dot', 0.6), output_var('F_sway', 30.0))
position_fis.print_rules()

**Watch the universe!** The trolley starts 5 m from the target, but `x_error` only
spans ±3 m. Outside its universe every term has membership 0, so *no rule fires* and
the controller would output nothing. We therefore **saturate** each input to its
variable's range before evaluating the FIS - a standard trick in fuzzy control.

In [ ]:
def saturate(fis, values):
    """Clamp each crisp input to its variable's universe of discourse."""
    return {name: torch.tensor(float(v)).clamp(fis.input_variables[name].range_min,
                                               fis.input_variables[name].range_max)
            for name, v in values.items()}

class FuzzyCrane:
    """obs = [x_err, x_err_dot, theta_err, theta_err_dot] -> [force]."""
    def __init__(self, position_fis, sway_fis, sway_gain=-0.5, max_force=30.0):
        self.pos, self.sway = position_fis, sway_fis
        self.sway_gain, self.max_force = sway_gain, max_force

    def __call__(self, obs):
        ex, dex, et, det = obs
        with torch.no_grad():
            f = self.pos(saturate(self.pos, {'x_error': ex, 'x_error_dot': dex})) \
                + self.sway_gain * self.sway(saturate(self.sway, {'theta_error': et,
                                                                  'theta_error_dot': det}))
        return np.array([float(f.clamp(-self.max_force, self.max_force))], dtype=np.float32)

fuzzy_crane = FuzzyCrane(position_fis, sway_fis)
fuzzy_log = crane_rollout(STEP, fuzzy_crane)
plot_crane({'state feedback': base_log, 'fuzzy': fuzzy_log}, 'Crane step response - baseline vs fuzzy')

show_table({'state feedback': crane_report(base_log),
              'fuzzy': crane_report(fuzzy_log)})

Look at the position surface of our fuzzy controller - it is a smooth, saturating
version of a PD law:

In [ ]:
n = 60
ex, dex = torch.meshgrid(torch.linspace(-3, 3, n), torch.linspace(-1.5, 1.5, n), indexing='xy')
with torch.no_grad():
    Fz = position_fis({'x_error': ex.reshape(-1), 'x_error_dot': dex.reshape(-1)}).reshape(n, n)
plt.figure(figsize=(5.5, 4.2))
plt.contourf(ex, dex, Fz, levels=20, cmap='coolwarm'); plt.colorbar(label='F_pos [N]')
plt.xlabel('x_error [m]'); plt.ylabel('x_error_dot [m/s]'); plt.title('position FIS surface');

### Disturbance rejection

A second scenario holds the setpoint and hits the trolley with a 25 N impulse at
t = 10 s. Good controllers recover quickly without large sway.

In [ ]:
DIST = disturbance_rejection(1500, setpoint=[5.0, 0.0], time=10.0, magnitude=25.0, duration=0.2)
logs = {'state feedback': crane_rollout(DIST, baseline), 'fuzzy': crane_rollout(DIST, fuzzy_crane)}
plot_crane(logs, 'Crane disturbance rejection (25 N impulse at t = 10 s)')
step0 = int(10.0 / CranePlant.default_dt)
show_table({name: disturbance_metrics(torch.tensor(log['x']), target=5.0,
                                        dt=CranePlant.default_dt, disturbance_step=step0)
              for name, log in logs.items()})

With just 2 × 9 readable rules, the fuzzy controller settles faster than the baseline
with roughly a third of the sway, at the price of a higher IAE: it accelerates more
gently, using about a quarter of the control effort. It also rejects the impulse below
better.

**Your turn:** try widening/narrowing the `span`s, changing `sway_gain`, or replacing
the rule table. Which settings trade speed against sway?

## 4. Fuzzy Grand Prix

A dynamic kart races on a closed circuit. There are three circuits:

In [ ]:
from fuzzy_torch.envs import make_track

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, name in zip(axes, ('oval', 'chicane', 'technical')):
    track = make_track(name)
    pts = np.vstack([track.points, track.points[:1]])
    ax.plot(pts[:, 0], pts[:, 1], color='#999', lw=14, solid_capstyle='round')
    ax.plot(pts[:, 0], pts[:, 1], 'w--', lw=1)
    ax.plot(*track.points[0], 'k>', ms=10)
    ax.set_aspect('equal'); ax.set_title(f'{name}  ({track.length:.0f} m)')
plt.tight_layout()

The observation is **track-relative**, so the same rule base works on every circuit:

`[cross_track_error, heading_error, speed, yaw_rate, lookahead_curvature, progress]`

The action is `[throttle, steering]` in `[-1, 1]`. A lap ends at the finish line;
leaving the track is a **DNF** (did not finish), so corners cannot be cut.

The library ships a reference `FuzzyRaceDriver`. Let's race it on every circuit:

In [ ]:
def race(driver, track='oval'):
    env = RacingEnv(track)
    obs, _ = env.reset(seed=0)
    path, speeds = [env._state[0, :2].numpy().copy()], [0.0]
    while True:
        obs, _, terminated, truncated, info = env.step(driver(obs))
        path.append(env._state[0, :2].numpy().copy())
        speeds.append(info['speed'])
        if terminated or truncated:
            break
    return env.track, np.asarray(path), np.asarray(speeds), info


def plot_race(results, title):
    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5), squeeze=False)
    for ax, (name, (track, path, speeds, info)) in zip(axes[0], results.items()):
        pts = np.vstack([track.points, track.points[:1]])
        ax.plot(pts[:, 0], pts[:, 1], color='#ccc', lw=14, solid_capstyle='round')
        sc = ax.scatter(path[:, 0], path[:, 1], c=speeds, s=3, cmap='plasma')
        lap = f"{info['lap_time']:.2f} s" if info['lap_time'] else 'DNF'
        ax.set_title(f'{name}: {lap}'); ax.set_aspect('equal')
    fig.colorbar(sc, ax=axes[0].tolist(), label='speed [m/s]')
    fig.suptitle(title)

ref = {t: race(FuzzyRaceDriver(), t) for t in ('oval', 'chicane', 'technical')}
plot_race(ref, 'Reference FuzzyRaceDriver')
for t, (*_, info) in ref.items():
    print(f"{t:10s} lap_time={info['lap_time']}  off_track={info['off_track']}")

### Drivers as JSON

For the live class leaderboard (`examples/race_control/`) a driver is just a JSON file
with two fuzzy systems - `throttle` and `steering` - in the format from notebook 01.
Each system may read any of the observation names above (plus `curve`, the absolute
curvature). `ConfigRaceDriver` turns the JSON into a driver.

In [ ]:
example_spec = json.loads((REPO / 'examples/race_control/example_driver.json').read_text())
print('systems:', list(example_spec))
print('steering inputs:', list(example_spec['steering']['inputs']))
print('throttle rules :', len(example_spec['throttle']['rules']))

json_driver = ConfigRaceDriver(example_spec)
res = {t: race(json_driver, t) for t in ('oval', 'chicane', 'technical')}
plot_race(res, 'example_driver.json')

### Write your own driver

A minimal driver: steer against the cross-track and heading errors, and lift off the
throttle when the road ahead curves. Edit the terms and rules, then re-run.

In [ ]:
my_driver_spec = {
    "steering": {
        "inputs": {
            "heading_error": {"range": [-3.15, 3.15], "terms": {
                "left":     ["trap", -3.15, -3.15, -0.4, 0.0],
                "straight": ["tri", -0.4, 0.0, 0.4],
                "right":    ["trap", 0.0, 0.4, 3.15, 3.15]}},
            "cross_track_error": {"range": [-1.0, 1.0], "terms": {
                "left":   ["trap", -1, -1, -0.3, 0.0],
                "centre": ["tri", -0.3, 0.0, 0.3],
                "right":  ["trap", 0.0, 0.3, 1, 1]}}
        },
        "output": {"range": [-1, 1], "terms": {
            "hard_right": ["tri", -1, -1, -0.5], "right": ["tri", -1, -0.5, 0],
            "none": ["tri", -0.5, 0, 0.5],
            "left": ["tri", 0, 0.5, 1], "hard_left": ["tri", 0.5, 1, 1]}},
        "rules": [
            {"if": {"heading_error": "left"},  "then": "hard_right"},
            {"if": {"heading_error": "right"}, "then": "hard_left"},
            {"if": {"heading_error": "straight", "cross_track_error": "left"},   "then": "left"},
            {"if": {"heading_error": "straight", "cross_track_error": "centre"}, "then": "none"},
            {"if": {"heading_error": "straight", "cross_track_error": "right"},  "then": "right"}
        ]
    },
    "throttle": {
        "inputs": {"curve": {"range": [0, 1.5], "terms": {
            "straight": ["tri", 0, 0, 0.3],
            "bend":     ["trap", 0.1, 0.4, 1.5, 1.5]}}},
        "output": {"range": [0, 1], "terms": {
            "slow": ["tri", 0, 0.3, 0.6], "fast": ["tri", 0.5, 1, 1]}},
        "rules": [
            {"if": {"curve": "straight"}, "then": "fast"},
            {"if": {"curve": "bend"},     "then": "slow"}
        ]
    }
}

mine = ConfigRaceDriver(my_driver_spec)
res = {t: race(mine, t) for t in ('oval', 'chicane', 'technical')}
plot_race(res, 'my driver')

> **Sign check.** If your kart spins out immediately, flip the steering signs:
> print a few observations and actions to see which way "positive" turns.
>
> To join the class leaderboard, save the spec with
> `json.dump(my_driver_spec, open('my_driver.json', 'w'), indent=2)` and submit it at
> the URL your instructor gives you (the server is `examples/race_control/server.py`).

## 5. Quadrotor

A 13-state rigid-body quadrotor: world position and velocity, a unit quaternion and
body angular rates. Two action modes are available:

- `ControlMode.MOTOR_THRUST`: four rotor thrusts [N] (the physical actuators),
- `ControlMode.RATE_THRUST`: collective thrust + roll / pitch / yaw rate requests
  (what a flight controller usually exposes - easier for fuzzy design).

First: hovering. Each rotor must carry a quarter of the weight.

In [ ]:
drone = QuadrotorPlant(mode=ControlMode.MOTOR_THRUST)
env = FuzzyControlEnv(drone)
obs, _ = env.reset(seed=0)
hover = [float(drone.m * drone.g / 4)] * 4
z = []
for _ in range(200):
    obs, *_ = env.step(hover)
    z.append(obs[2])
print(f'mass {drone.m} kg -> hover thrust per rotor {hover[0]:.3f} N')
print(f'altitude drift after {200 * env.dt:.1f} s: {z[-1] - z[0]:+.2e} m')

Now control the altitude with the `DroneStarter`: a factorised fuzzy controller made of
four small TSK systems (`roll`, `pitch`, `vertical`, `yaw`) from the student workspace
(section 6). Its consequents start at **zero**, so out of the box it only hovers.

We give the `vertical` system a first-order consequent in every rule,
`f = k_e · z_error + k_v · z_velocity`, i.e. a fuzzy PD altitude controller, and ask it to
climb from the ground to 1.8 m.

In [ ]:
sys.path.insert(0, str(REPO / 'examples' / 'student'))
from fuzzy_starters import CraneStarter, RaceStarter, DroneStarter

def set_linear(system, gains):
    """Initialise every TSK rule of `system` as f = sum(gain_i * input_i).

    The consequents normalise each input onto [-1, 1], so each raw gain is multiplied
    by the half-range of its input variable.
    """
    halves = [(v.range_max - v.range_min) / 2 for v in system.input_variables.values()]
    coeffs = torch.tensor([0.0] + [g * h for g, h in zip(gains, halves)])
    with torch.no_grad():
        for rule in system.rules:
            rule.consequent_fn.coefficients.copy_(coeffs)

def fly(controller, target=(0.0, 0.0, 1.8), seconds=8.0, seed=7):
    drone = QuadrotorPlant(mode=ControlMode.RATE_THRUST, target=target)
    scenario = Scenario(name='climb', episode_steps=round(seconds / drone.default_dt),
                        setpoint=list(drone.target))
    env = FuzzyControlEnv(drone, scenario=scenario)
    env.reset(seed=seed)
    t, pos, thrust = [], [], []
    for _ in range(scenario.episode_steps):
        state = env._state[0].detach().cpu().numpy()
        target_ = env.setpoint().detach().cpu().numpy()
        _, _, _, truncated, info = env.step(controller.act(state, target_))
        t.append(info['time']); thrust.append(info['control'][0])
        pos.append(env._state[0, :3].detach().cpu().numpy())
        if truncated:
            break
    return np.asarray(t), np.asarray(pos), np.asarray(thrust), np.asarray(target)

pilot = DroneStarter()
print('sub-systems:', [n for n, _ in pilot.named_children()])
results = {}
for gains in [(0.0, 0.0), (1.0, 0.5), (3.0, 2.5)]:
    p = DroneStarter()
    set_linear(p.vertical, gains)
    p.eval()
    results[gains] = fly(p)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for (ke, kv), (t, pos, thrust, target) in results.items():
    axes[0].plot(t, pos[:, 2], label=f'k_e={ke}, k_v={kv}')
    axes[1].plot(t, thrust, label=f'k_e={ke}, k_v={kv}')
axes[0].axhline(target[2], color='k', ls='--', lw=1, label='target')
axes[0].set(xlabel='time [s]', ylabel='altitude z [m]')
axes[1].set(xlabel='time [s]', ylabel='collective thrust [N]')
axes[0].legend(fontsize=8)
fig.suptitle('Quadrotor altitude with the DroneStarter vertical FIS')
plt.tight_layout()
for gains, (t, pos, thrust, target) in results.items():
    print(f'gains {gains}: final altitude error {abs(pos[-1, 2] - target[2]):.4f} m')

**Why only altitude?** The lateral `roll`/`pitch` systems map position error and
velocity straight to an angular **rate**. Without attitude feedback the tilt keeps
integrating and the drone flips over. Try it with `set_linear(p.pitch, (0.6, 0.8))` and a
target of `(1, 0, 1.8)`. Fixing this (for example by adding the tilt angle as an input,
like the cascaded controller in `examples/drone_demo.py`) is one of the coursework tasks.

## 6. The student workspace

`examples/student/fuzzy_starters.py` is **the only file you edit** for the coursework.
The plants, rewards and scenarios are fixed so that everybody's results are
comparable. It provides three starters with fixed input/output contracts:

| case | inputs | action | report |
|---|---|---|---|
| Crane | `[x_error, x_error_dot, theta_error, theta_error_dot]` | trolley force [N] | IAE, residual sway, control effort |
| Kart | steering: `[cross_track_error, heading_error]`; speed: `[abs(curvature), speed]` | `[throttle, steering]` | lap time, or DNF |
| Drone | position/velocity errors, attitude and body rates | `[collective_thrust, roll_rate, pitch_rate, yaw_rate]` | final and integrated 3-D position error |

Each starter is a grid of first-order TSK rules with 3 Gaussian terms per input and
**zeroed consequents** - a safe neutral controller that hides no solution.

In [ ]:
for cls in (CraneStarter, RaceStarter, DroneStarter):
    c = cls()
    n_rules = sum(len(m.rules) for m in c.modules() if isinstance(m, FuzzyModule))
    n_params = sum(p.numel() for p in c.parameters())
    print(f'{cls.__name__:13s} rules={n_rules:3d}  trainable parameters={n_params}')

print('\nCraneStarter on a sample observation:', CraneStarter()(np.array([5.0, 0.0, 0.1, 0.0])))

### Training a starter from data

The starters are `nn.Module`s, so they train like the systems in notebook 01. As a
first exercise we **imitate** the state-feedback baseline: collect the observations it
visits (plus some random perturbations), record its actions, and fit the 81-rule
`CraneStarter` to them with a batched regression. This is quick, because the whole
dataset is evaluated in one call. *(About a minute on a laptop CPU.)*

In [ ]:
rng = np.random.default_rng(0)
env = ErrorFeatures(FuzzyControlEnv(CranePlant(), scenario=STEP))
obs, _ = env.reset(seed=0)
visited = []
for _ in range(STEP.episode_steps):
    visited.append(obs)
    obs, *_ = env.step(baseline(obs))
visited = np.asarray(visited)
X = np.vstack([visited, visited + rng.normal(size=visited.shape) * [0.5, 0.3, 0.05, 0.2]])
Y = np.array([baseline(x)[0] for x in X])
X, Y = torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)
print('dataset:', tuple(X.shape))

torch.manual_seed(0)
student = CraneStarter()
student.train()
opt = torch.optim.Adam(student.parameters(), lr=0.15)
losses = []
for epoch in range(150):
    opt.zero_grad()
    loss = ((student(X) - Y) ** 2).mean()     # batched forward: (N, 4) -> (N,)
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(f'MSE: {losses[0]:.2f} -> {losses[-1]:.2f}')

In [ ]:
student.eval()
imitated_log = crane_rollout(STEP, student)
plot_crane({'state feedback': base_log, 'hand fuzzy': fuzzy_log,
            'CraneStarter (imitation)': imitated_log}, 'Crane: three controllers')
show_table({'state feedback': crane_report(base_log), 'hand fuzzy': crane_report(fuzzy_log),
            'CraneStarter (imitation)': crane_report(imitated_log)})

Imitation can only copy the teacher. To *improve* on it, optimise a closed-loop
cost such as IAE plus sway. Because the plants are differentiable torch code, you can
backpropagate that cost through a simulated trajectory (`fuzzy_torch.envs.rk4_step`
or `rollout`, on the batched plant). This is far more expensive than imitation, so
start with short horizons and small rule bases.

## 7. Running the demos and tests

From the repository root:

```bash
pip install -e ".[viz,test]"

python examples/crane_demo.py --out out/          # plots + animated GIF
python examples/crane_demo.py --live              # interactive 3-D viewer
python examples/racing_demo.py --live --backend mujoco
python examples/drone_demo.py --live

python examples/race_control/server.py            # class leaderboard (see its README)

pytest                                            # the test-suite
```

## 8. Exercises

1. **Crane.** Tune the hand-written fuzzy crane to match the baseline's IAE while
   keeping its low sway. Report IAE, settling time, maximum sway and control effort.
2. **Rule table.** Replace the 3×3 table with a 5×5 one (NB, NS, Z, PS, PB inputs).
   Is the extra complexity worth it?
3. **Grand Prix.** `my_driver_spec` finishes every circuit but is slower than the
   reference driver. Add `speed` as a throttle input so the kart brakes *before* corners
   and beat the reference lap times. Submit your driver to the leaderboard.
4. **Drone.** Make the `DroneStarter` fly to `(1.4, -1.0, 1.8)`. You will need
   attitude information in the lateral systems (see the note in section 5).
5. **Learning.** Fit the `RaceStarter` by imitating `FuzzyRaceDriver` on the oval, then
   test it on the `technical` circuit. Does imitation generalise to an unseen track?